# Tech Challenge — Fase 3: Full Fine-Tuning e assistente médico

**Objetivo:** documentar a preparação do MedQuAD, o Full Fine-Tuning definitivo do Qwen e sua integração a um assistente médico demonstrativo com RAG e LangGraph.

**Modelo:** Qwen2.5-0.5B-Instruct · **Dados:** MedQuAD e prontuários sintéticos · **Ambiente:** Google Colab · **Uso:** exclusivamente acadêmico


## 1. Ambiente e carregamento do MedQuAD

### Verificação do ambiente

Confirmamos a disponibilidade de CUDA e o modelo da GPU. A GPU acelera operações matriciais necessárias ao treinamento.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
CUDA disponível: False


### Instalação de dependências

Instalamos bibliotecas para modelos Transformers, datasets e treinamento.

In [ ]:
%pip install -q transformers datasets accelerate

### Conferência das versões

Registramos as versões para facilitar a reprodução e o diagnóstico de incompatibilidades.

In [ ]:
import transformers
import datasets
import accelerate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0


### Montagem do Google Drive

O armazenamento do Colab é temporário. Montar o Drive permite acessar os JSONL processados e salvar os pesos treinados.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Carregamento do MedQuAD processado

Carregamos `train.jsonl` e `val.jsonl`. O conjunto de teste foi separado previamente e não é utilizado no treinamento. Os campos esperados são `instruction` (pergunta) e `response` (resposta).


In [ ]:
from datasets import load_dataset

base_path = "/content/drive/MyDrive/TechChallenge3/data/processed"

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{base_path}/train.jsonl",
        "val": f"{base_path}/val.jsonl"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 13076
    })
    val: Dataset({
        features: ['instruction', 'response'],
        num_rows: 1645
    })
})


### Inspeção de um registro

Verificamos se a estrutura dos dados corresponde ao esperado antes de iniciar o processamento.

In [ ]:
example = dataset["train"][0]

print("Pergunta:", example["instruction"])
print("Resposta:", example["response"][:500])

Pergunta: Is primary myelofibrosis inherited ?
Resposta: This condition is generally not inherited but arises from gene mutations that occur in early blood-forming cells after conception. These alterations are called somatic mutations.


## 2. Preparação dos dados para o Fine-Tuning

### Modelo base e tokenizer

Usamos `Qwen/Qwen2.5-0.5B-Instruct`. **Full Fine-Tuning** atualiza todos os pesos treináveis.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Carrega o tokenizer, responsável por converter texto em tokens.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Carrega o modelo base em FP32
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto"
)

# Full Fine-Tuning: não congelamos nenhuma camada.
model.requires_grad_(True)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Modelo carregado com sucesso!")
print("Dispositivo:", next(model.parameters()).device)
print("Precisão dos pesos:", next(model.parameters()).dtype)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado com sucesso!
Dispositivo: cpu
Precisão dos pesos: torch.float32


### Conferência dos parâmetros treináveis

Somamos os parâmetros do modelo e verificamos que todos estão habilitados para treinamento.

In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Parâmetros totais: {total_params:,}")
print(f"Percentual treinável: {100 * trainable_params / total_params:.2f}%")

Parâmetros treináveis: 494,032,768
Parâmetros totais: 494,032,768
Percentual treinável: 100.00%


### Formatação no padrão de conversa

O *chat template* converte pergunta e resposta em uma sequência com papéis `user` e `assistant`, seguindo o formato esperado pelo Qwen.

In [ ]:
# Converte cada par pergunta/resposta no formato conversacional do Qwen.
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    # Não acrescenta um prompt de geração: a resposta já está presente.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

### Aplicação da formatação

Aplicamos a função a todos os registros e criamos uma coluna `text` sem descartar os campos originais.

In [ ]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)

Map:   0%|          | 0/13076 [00:00<?, ? examples/s]

Map:   0%|          | 0/1645 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'text'],
        num_rows: 13076
    })
    val: Dataset({
        features: ['instruction', 'response', 'text'],
        num_rows: 1645
    })
})


### Inspeção do texto formatado

Conferimos os marcadores especiais da conversa para garantir que a entrada segue o formato do modelo.

In [ ]:
print(formatted_dataset["train"][0]["text"][:1200])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Is primary myelofibrosis inherited ?<|im_end|>
<|im_start|>assistant
This condition is generally not inherited but arises from gene mutations that occur in early blood-forming cells after conception. These alterations are called somatic mutations.<|im_end|>



### Filtragem por comprimento

Selecionamos exemplos de até 512 tokens para evitar truncamento durante o Full Fine-Tuning definitivo. Exemplos mais longos são excluídos.


In [ ]:
# Limite adotado no treinamento definitivo; exemplos longos são excluídos.
MAX_LENGTH = 512

def fits_max_length(example):
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )

    return len(tokens["input_ids"]) <= MAX_LENGTH

### Construção dos labels e máscara de perda

A pergunta é contexto, não alvo de previsão. Marcamos seus tokens com `-100` para que não contribuam para a *loss*. Os tokens da resposta permanecem como alvos. Conferimos o alinhamento dos tokens.

In [ ]:
def tokenize_example(example):
    text = example["text"]

    # Marcador que identifica o início da resposta
    assistant_marker = "<|im_start|>assistant\n"

    if assistant_marker not in text:
        raise ValueError("Marcador do assistente não encontrado")

    # Separa o texto antes da resposta
    prefix, _ = text.split(assistant_marker, 1)
    prefix += assistant_marker

    # Tokeniza a conversa completa
    full_tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    # Tokeniza apenas o prefixo
    prefix_tokens = tokenizer(
        prefix,
        add_special_tokens=False
    )["input_ids"]

    # Confere se os tokens estão alinhados
    if full_tokens[:len(prefix_tokens)] != prefix_tokens:
        raise ValueError("Os tokens do prefixo não estão alinhados")

    prefix_length = len(prefix_tokens)

    # Os labels começam como cópia dos IDs da conversa inteira.
    labels = full_tokens.copy()

    # -100 indica ao cálculo da perda que a pergunta não deve ser prevista.
    labels[:prefix_length] = [-100] * prefix_length

    return {
        "input_ids": full_tokens,
        "attention_mask": [1] * len(full_tokens),
        "labels": labels
    }

### Validação dos exemplos tokenizados

Verificamos comprimentos, limite máximo, presença de tokens supervisionados e correspondência entre labels e IDs.

In [ ]:
def validate_tokenized_dataset(data, max_length=512):
    for index, example in enumerate(data):
        input_ids = example["input_ids"]
        labels = example["labels"]
        attention_mask = example["attention_mask"]

        assert len(input_ids) == len(labels) == len(attention_mask), (
            f"Tamanhos incompatíveis para o exemplo {index}"
        )

        assert len(input_ids) <= max_length, (
            f"O exemplo {index} ultrapassou o limite"
        )

        assert any(label != -100 for label in labels), (
            f"O exemplo {index} não possui tokens de resposta"
        )

        assert all(
            label == -100 or label == token
            for label, token in zip(labels, input_ids)
        ), f"Labels incorretos no exemplo {index}"

    print(f"Validação concluída: {len(data)} exemplos corretos!")


### Data collator: padding dinâmico

Cada lote pode reunir sequências de comprimentos diferentes. Adicionamos tokens de padding ao final; a máscara de atenção vale `0` nesses tokens, e os labels de padding recebem `-100`.

In [ ]:
import torch

def data_collator(features):
    # O maior exemplo define o comprimento do lote, evitando padding fixo.
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for feature in features:
        # O padding não é supervisionado (label=-100).
        padding_length = max_length - len(feature["input_ids"])

        input_ids.append(
            feature["input_ids"]
            + [tokenizer.pad_token_id] * padding_length
        )

        attention_masks.append(
            feature["attention_mask"]
            + [0] * padding_length
        )

        labels.append(
            feature["labels"]
            + [-100] * padding_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

### Preparação do Full Fine-Tuning ampliado

Preparamos o conjunto definitivo de treinamento e validação, mantendo somente os exemplos de até 512 tokens.

O conjunto de validação acompanha as métricas do treinamento. O conjunto de teste permanece reservado para avaliação independente.

O treinamento definitivo será iniciado a partir dos pesos originais do Qwen2.5-0.5B-Instruct.


In [ ]:
# Recupera todos os exemplos formatados.
full_train = formatted_dataset["train"]
full_val = formatted_dataset["val"]

# Aplica o filtro de comprimento definido na preparação.
# Exemplos acima de 512 tokens são excluídos, não truncados.
full_train_filtered = full_train.filter(fits_max_length)
full_val_filtered = full_val.filter(fits_max_length)

print("Treinamento disponível:", len(full_train))
print("Treinamento elegível:", len(full_train_filtered))
print("Validação disponível:", len(full_val))
print("Validação elegível:", len(full_val_filtered))

# Tokeniza os conjuntos elegíveis, preservando a máscara
# que impede o cálculo da loss sobre a pergunta.
full_train_tokenized = full_train_filtered.map(
    tokenize_example,
    remove_columns=full_train_filtered.column_names
)

full_val_tokenized = full_val_filtered.map(
    tokenize_example,
    remove_columns=full_val_filtered.column_names
)

# Verifica se a tokenização produziu exemplos válidos.
validate_tokenized_dataset(full_train_tokenized)
validate_tokenized_dataset(full_val_tokenized)

print("\nDados preparados para o treinamento ampliado!")

Filter:   0%|          | 0/13076 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1645 [00:00<?, ? examples/s]

Treinamento disponível: 13076
Treinamento elegível: 11391
Validação disponível: 1645
Validação elegível: 1439


Map:   0%|          | 0/11391 [00:00<?, ? examples/s]

Map:   0%|          | 0/1439 [00:00<?, ? examples/s]

Validação concluída: 11391 exemplos corretos!
Validação concluída: 1439 exemplos corretos!

Dados preparados para o treinamento ampliado!


## 3. Full Fine-Tuning definitivo e exportação

### Inicialização do modelo para Full Fine-Tuning

Liberamos a memória da GPU e carregamos o Qwen2.5-0.5B-Instruct a partir dos pesos originais.

Todos os parâmetros continuam treináveis, com pesos FP32 e *gradient checkpointing* habilitado para reduzir o uso de memória.


In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM

# Libera referências anteriores para inicializar o treinamento definitivo.
for name in ["trainer", "model", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Confirma que o Colab continua com GPU disponível.
assert torch.cuda.is_available(), (
    "GPU indisponível. Verifique o ambiente do Colab."
)

print("GPU:", torch.cuda.get_device_name(0))

# Carrega os pesos originais em FP32.
# O FP16 será usado pelo Trainer em precisão mista.
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float32,
    device_map="auto"
)

# Full Fine-Tuning: todos os parâmetros serão ajustados.
model.requires_grad_(True)

# Reduz o consumo de memória durante o treinamento.
model.config.use_cache = False
model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Parâmetros totais:", total_params)
print("Parâmetros treináveis:", trainable_params)
print("Tipo dos pesos:", next(model.parameters()).dtype)
print("Dispositivo:", next(model.parameters()).device)

### Configuração do treinamento ampliado

Configuramos o Full Fine-Tuning definitivo com 11.391 exemplos elegíveis, durante uma época.

Utilizamos AdamW, taxa de aprendizado de 2e-5, acumulação de gradientes de 8 etapas e precisão mista FP16. As perdas e os checkpoints são registrados para acompanhamento e recuperação do treinamento.


In [ ]:
import os
from pathlib import Path
from transformers import TrainingArguments, Trainer

# Diretório de checkpoints do treinamento definitivo.
checkpoint_path = (
    "/content/checkpoints/qwen-medquad-full-ft-v2"
)

# Evita iniciar acidentalmente outro treinamento sobre
# checkpoints já existentes.
existing_checkpoints = list(
    Path(checkpoint_path).glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Já existem checkpoints neste diretório. "
        "Verifique a retomada antes de iniciar outro treino."
    )

training_args_v2 = TrainingArguments(
    output_dir=checkpoint_path,

    # Treinamento completo: uma passagem pelos dados.
    num_train_epochs=1,
    max_steps=-1,

    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Acompanhamento das métricas.
    logging_strategy="steps",
    logging_steps=25,

    # Avaliação ao longo da época.
    eval_strategy="steps",
    eval_steps=700,

    # Checkpoints para recuperar o progresso.
    save_strategy="steps",
    save_steps=700,
    save_total_limit=1,
    save_only_model=False,

    # Otimização de memória.
    gradient_checkpointing=True,
    fp16=True,
    optim="adamw_torch",

    # Reprodutibilidade e configuração do Trainer.
    seed=42,
    data_seed=42,
    report_to="none",
    remove_unused_columns=False
)

trainer_v2 = Trainer(
    model=model,
    args=training_args_v2,
    train_dataset=full_train_tokenized,
    eval_dataset=full_val_tokenized,
    data_collator=data_collator
)

print("Trainer do Full Fine-Tuning configurado.")
print("Exemplos de treino:", len(full_train_tokenized))
print("Exemplos de validação:", len(full_val_tokenized))
print("Épocas:", training_args_v2.num_train_epochs)
print("Checkpoint:", checkpoint_path)

Trainer do segundo experimento configurado.
Exemplos de treino: 11391
Exemplos de validação: 1439
Épocas: 1
Checkpoint: /content/checkpoints/qwen-medquad-full-ft-v2


### Execução do Full Fine-Tuning ampliado
Executamos uma época de treinamento com os dados elegíveis do MedQuAD.

Acompanhamos a loss de treinamento e de validação para investigar o aprendizado e possíveis sinais de sobreajuste.

Ao final, salvamos o modelo completo e registramos as métricas observadas. A melhoria da precisão médica será investigada posteriormente por meio de avaliação comparativa.

In [ ]:
import json
import os

# Inicia o treinamento ampliado.
train_result_v2 = trainer_v2.train()

print("\nTreinamento ampliado concluído!")
print("Métricas:", train_result_v2.metrics)

# Diretório definitivo dos pesos ajustados.
final_model_path = (
    "/content/models/qwen-medquad-full-ft-v2"
)

os.makedirs(final_model_path, exist_ok=True)

# Salva os pesos completos e o tokenizer.
trainer_v2.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Recupera a última avaliação registrada pelo Trainer.
eval_logs = [
    entry for entry in trainer_v2.state.log_history
    if "eval_loss" in entry
]

last_evaluation = eval_logs[-1] if eval_logs else None

# Registra os resultados para o relatório e o vídeo.
metrics = {
    "training": train_result_v2.metrics,
    "last_evaluation": last_evaluation,
    "training_examples": len(full_train_tokenized),
    "validation_examples": len(full_val_tokenized),
    "epochs_requested": 1
}

metrics_file = os.path.join(
    final_model_path,
    "training_metrics.json"
)

with open(metrics_file, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, default=str)

print("\nModelo salvo em:", final_model_path)
print("Última avaliação:", last_evaluation)
print("Métricas salvas em:", metrics_file)

Step,Training Loss,Validation Loss
700,1.141652,1.295597
1400,1.159428,1.216180
1424,1.159428,1.215799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Treinamento ampliado concluído!
Métricas: {'train_runtime': 3735.3176, 'train_samples_per_second': 3.05, 'train_steps_per_second': 0.381, 'total_flos': 5497177567444992.0, 'train_loss': 1.2814904746044888, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo salvo em: /content/models/qwen-medquad-full-ft-v2
Última avaliação: {'eval_loss': 1.2157994508743286, 'eval_runtime': 66.834, 'eval_samples_per_second': 21.531, 'eval_steps_per_second': 21.531, 'epoch': 1.0, 'step': 1424}
Métricas salvas em: /content/models/qwen-medquad-full-ft-v2/training_metrics.json


### Exportação e preservação do modelo ajustado
Após concluir o Full Fine-Tuning ampliado, verificamos se os arquivos do modelo foram salvos corretamente.

Em seguida, criamos um arquivo ZIP contendo os pesos ajustados, o tokenizer, a configuração e as métricas.

Esse arquivo será transferido para o computador local, evitando a perda do treinamento quando a sessão temporária do Google Colab for encerrada.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

# Diretório onde salvamos o modelo treinado.
model_dir = Path(
    "/content/models/qwen-medquad-full-ft-v2"
)

# Verifica os arquivos essenciais.
assert model_dir.is_dir(), "Diretório do modelo não encontrado."
assert (model_dir / "config.json").exists(), (
    "Configuração do modelo não encontrada."
)

weight_files = list(model_dir.glob("*.safetensors"))

assert weight_files, "Pesos do modelo não encontrados."

# Mostra os arquivos presentes e seus tamanhos.
print("Arquivos do modelo:")
for file in sorted(model_dir.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"- {file.name}: {size_mb:.2f} MB")

# Cria ZIP sem recomprimir os pesos.
# Isso evita processamento desnecessário.
zip_path = Path(
    "/content/qwen-medquad-full-ft-v2.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True
) as archive:

    for file in model_dir.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(model_dir.parent)
            )

print("\nArquivo ZIP criado!")
print(
    "Tamanho:",
    round(zip_path.stat().st_size / (1024 ** 3), 2),
    "GB"
)

# Solicita o download para seu computador.
files.download(str(zip_path))

Arquivos do modelo:
- chat_template.jinja: 0.00 MB
- config.json: 0.00 MB
- generation_config.json: 0.00 MB
- model.safetensors: 1884.62 MB
- tokenizer.json: 10.89 MB
- tokenizer_config.json: 0.00 MB
- training_args.bin: 0.00 MB
- training_metrics.json: 0.00 MB

Arquivo ZIP criado!
Tamanho: 1.85 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Prontuários sintéticos e RAG

### Preparação do ambiente do assistente médico

Nesta etapa, preparamos as bibliotecas necessárias para implementar
o agente RAG, o banco vetorial ChromaDB e a orquestração LangGraph.

Utilizaremos o Google Colab em CPU, sem repetir o treinamento
do Qwen realizado anteriormente.

O Google Drive armazenará os dados sintéticos e os resultados
do projeto.

In [18]:
import sys
import subprocess
import importlib.util
from pathlib import Path

# Instala somente bibliotecas que ainda não estão disponíveis.
dependencias = {
    "langchain_chroma": "langchain-chroma",
    "langchain_huggingface": "langchain-huggingface",
    "langchain_core": "langchain-core",
    "langgraph": "langgraph",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers"
}

faltando = [
    pacote
    for modulo, pacote in dependencias.items()
    if importlib.util.find_spec(modulo) is None
]

if faltando:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--quiet", *faltando
    ])

# Monta o Google Drive.
from google.colab import drive

drive.mount("/content/drive")

# Define a pasta principal do projeto.
BASE_PATH = Path(
    "/content/drive/MyDrive/TechChallenge3"
)

DATA_PATH = BASE_PATH / "data" / "synthetic"
RESULTS_PATH = BASE_PATH / "results"

DATA_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("[OK] Ambiente preparado.")
print("Diretório do projeto:", BASE_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Ambiente preparado.
Diretório do projeto: /content/drive/MyDrive/TechChallenge3


### Criação da base de pacientes sintéticos

Criamos 1.000 prontuários inteiramente fictícios para demonstrar
a recuperação de informações hospitalares.

Cada paciente possui dados cadastrais, observações clínicas
e exames com status simulados.

Utilizamos uma semente fixa para permitir a reprodução dos dados.

Os registros são exclusivamente acadêmicos e não representam
pacientes reais.

In [19]:
import json
import random

# Arquivo persistente da base sintética.
PATIENTS_FILE = DATA_PATH / "patients_vector_v1.json"

if PATIENTS_FILE.exists():
    # Reutiliza a base existente sem sobrescrever alterações.
    with open(PATIENTS_FILE, "r", encoding="utf-8") as file:
        pacientes = json.load(file)

    print("Base existente carregada.")

else:
    rng = random.Random(42)

    primeiros_nomes = [
        "Marina", "Carlos", "Ana", "Pedro", "Juliana",
        "Rafael", "Fernanda", "Lucas", "Camila", "Bruno",
        "Beatriz", "Gabriel", "Larissa", "Eduardo", "Renata"
    ]

    sobrenomes = [
        "Almeida", "Costa", "Silva", "Oliveira",
        "Santos", "Souza", "Ferreira", "Pereira",
        "Rodrigues", "Martins", "Carvalho", "Lima"
    ]

    setores = [
        "Clínica Médica", "Observação",
        "Ambulatório", "Ortopedia"
    ]

    observacoes = [
        "Paciente relata episódios de cefaleia e tontura.",
        "Paciente relata desconforto abdominal intermitente.",
        "Paciente relata cansaço e dificuldade para dormir.",
        "Paciente apresenta relato de dor muscular.",
        "Paciente compareceu para acompanhamento de rotina."
    ]

    pacientes = []

    for numero in range(1, 1001):
        patient_id = f"PAC-DEMO-{numero:03d}"

        nome = (
            f"{rng.choice(primeiros_nomes)} "
            f"{rng.choice(sobrenomes)} "
            f"{rng.choice(sobrenomes)}"
        )

        # Preserva o paciente que utilizamos nos testes anteriores.
        if numero == 1:
            nome = "Marina Almeida Costa"

        observacao = (
            "Paciente relata episódios de cefaleia e tontura."
            if numero == 1
            else rng.choice(observacoes)
        )

        paciente = {
            "patient_id": patient_id,
            "nome": nome,
            "idade": rng.randint(18, 85),
            "email": f"paciente.{numero:04d}@hospital.invalid",
            "telefone": f"(00) 90000-{numero:04d}",
            "setor": rng.choice(setores),
            "observacoes": [
                {
                    "id": f"NOTA-{numero:04d}",
                    "texto": observacao
                }
            ],
            "exames": [
                {
                    "id": f"EX-{numero:04d}-01",
                    "nome": "Hemograma",
                    "status": (
                        "PENDENTE"
                        if numero % 3 != 0
                        else "CONCLUÍDO"
                    )
                },
                {
                    "id": f"EX-{numero:04d}-02",
                    "nome": "Glicemia",
                    "status": "CONCLUÍDO"
                }
            ]
        }

        pacientes.append(paciente)

    # Salva a base para reutilização entre sessões.
    with open(PATIENTS_FILE, "w", encoding="utf-8") as file:
        json.dump(
            pacientes,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("Nova base sintética criada.")

# Verificações antes de continuar.
assert len(pacientes) == 1000
assert len({
    paciente["patient_id"] for paciente in pacientes
}) == 1000

print("Pacientes:", len(pacientes))
print("Arquivo:", PATIENTS_FILE)
print("Primeiro paciente:", pacientes[0]["nome"])
print("Último identificador:", pacientes[-1]["patient_id"])

Base existente carregada.
Pacientes: 1000
Arquivo: /content/drive/MyDrive/TechChallenge3/data/synthetic/patients_vector_v1.json
Primeiro paciente: Marina Almeida Costa
Último identificador: PAC-DEMO-1000


### Indexação dos prontuários no ChromaDB

Nesta etapa, convertemos os dados sintéticos dos pacientes em
documentos que serão armazenados em um banco vetorial.

Cada paciente possui três documentos: cadastro, observações
clínicas e exames.

Utilizamos embeddings multilíngues para representar
semanticamente os documentos.

O índice é armazenado no Google Drive e poderá ser
reutilizado sem gerar os embeddings novamente.

O controle de acesso será implementado na próxima etapa.

In [20]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Diretório persistente para o banco vetorial.
CHROMA_PATH = BASE_PATH / "data" / "chroma_pacientes_v1"

# Modelo de embeddings multilíngue.
# Não é o Qwen; sua função é converter textos em vetores.
embeddings = HuggingFaceEmbeddings(
    model_name=(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Conecta ao banco ou cria a coleção, se necessário.
vectorstore_pacientes = Chroma(
    collection_name="prontuarios_sinteticos_v1",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH)
)

documentos = []
ids = []

for paciente in pacientes:
    pid = paciente["patient_id"]

    # Documento 1: cadastro.
    cadastro = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Nome: {paciente['nome']}\n"
            f"Idade: {paciente['idade']}\n"
            f"Setor: {paciente['setor']}\n"
            f"E-mail fictício: {paciente['email']}\n"
            f"Telefone fictício: {paciente['telefone']}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "cadastro"
        }
    )

    # Documento 2: observações clínicas.
    notas_texto = "\n".join(
        f"{nota['id']}: {nota['texto']}"
        for nota in paciente["observacoes"]
    )

    notas = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Observações clínicas:\n{notas_texto}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "notas"
        }
    )

    # Documento 3: exames.
    exames_texto = "\n".join(
        f"{exame['id']}: {exame['nome']} "
        f"| Status: {exame['status']}"
        for exame in paciente["exames"]
    )

    exames = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Exames:\n{exames_texto}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "exames"
        }
    )

    documentos.extend([cadastro, notas, exames])

    ids.extend([
        f"cadastro::{pid}",
        f"notas::{pid}",
        f"exames::{pid}"
    ])

# Evita recalcular embeddings de registros já indexados.
ids_existentes = set(
    vectorstore_pacientes.get(include=[])["ids"]
)

pendentes = [
    (documento, doc_id)
    for documento, doc_id in zip(documentos, ids)
    if doc_id not in ids_existentes
]

print("Documentos esperados:", len(documentos))
print("Documentos para indexar:", len(pendentes))

# Processa em lotes para limitar o uso de memória.
tamanho_lote = 100

for inicio in range(0, len(pendentes), tamanho_lote):
    lote = pendentes[inicio:inicio + tamanho_lote]

    vectorstore_pacientes.add_documents(
        documents=[item[0] for item in lote],
        ids=[item[1] for item in lote]
    )

    print(
        f"Indexados: "
        f"{min(inicio + tamanho_lote, len(pendentes))}"
        f"/{len(pendentes)}"
    )

# Confirma a quantidade armazenada.
total_indexado = vectorstore_pacientes._collection.count()

assert total_indexado == 3000, (
    f"Esperados 3000 documentos, encontrados {total_indexado}"
)

print("\n[OK] Indexação concluída!")
print("Total de documentos:", total_indexado)
print("Banco persistido em:", CHROMA_PATH)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documentos esperados: 3000
Documentos para indexar: 0

[OK] Indexação concluída!
Total de documentos: 3000
Banco persistido em: /content/drive/MyDrive/TechChallenge3/data/chroma_pacientes_v1


### Recuperação de prontuários com controle de acesso

Implementamos uma função de recuperação dos registros armazenados
no ChromaDB, utilizando o identificador do paciente.

Antes de acessar os documentos, verificamos se o profissional
solicitante possui autorização.

Para a demonstração acadêmica, utilizamos permissões fictícias.

A consulta retorna os documentos de cadastro, observações
clínicas e exames, preservando suas respectivas fontes.

Este mecanismo não representa autenticação real de profissionais.

In [21]:
import re

# Permissões fictícias para os testes acadêmicos.
permissoes_demo = {
    ("clinico_demo_01", "PAC-DEMO-001"),
    ("clinico_demo_01", "PAC-DEMO-003"),
    ("clinico_demo_01", "PAC-DEMO-1000"),
    ("clinico_demo_02", "PAC-DEMO-002")
}


def recuperar_paciente(ator, patient_id):
    """
    Recupera os registros do paciente no ChromaDB
    somente após verificar a autorização.
    """

    # 1. Valida o identificador recebido.
    if not re.fullmatch(r"PAC-DEMO-\d{3,4}", patient_id):
        raise ValueError("Identificador de paciente inválido.")

    # 2. Verifica autorização ANTES da consulta.
    if (ator, patient_id) not in permissoes_demo:
        raise PermissionError(
            "Profissional não autorizado a acessar este paciente."
        )

    # 3. Busca os três documentos do paciente.
    ids = [
        f"cadastro::{patient_id}",
        f"notas::{patient_id}",
        f"exames::{patient_id}"
    ]

    resultado = vectorstore_pacientes.get(
        ids=ids,
        include=["documents", "metadatas"]
    )

    if len(resultado["ids"]) != 3:
        raise ValueError(
            "Prontuário incompleto ou não encontrado no ChromaDB."
        )

    # 4. Organiza os registros por tipo.
    registros = {}
    fontes = []

    for doc_id, texto, metadata in zip(
        resultado["ids"],
        resultado["documents"],
        resultado["metadatas"]
    ):
        # Verifica se o documento pertence ao paciente.
        if metadata["patient_id"] != patient_id:
            raise ValueError("Inconsistência de identificação.")

        tipo = metadata["record_type"]
        registros[tipo] = texto

        fontes.append({
            "documento": doc_id,
            "tipo": tipo,
            "paciente": patient_id
        })

    if set(registros) != {"cadastro", "notas", "exames"}:
        raise ValueError("Tipos de documentos inconsistentes.")

    return {
        "patient_id": patient_id,
        "registros": registros,
        "fontes": fontes
    }


# TESTE 1 — Acesso autorizado.
paciente_001 = recuperar_paciente(
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

assert "Marina Almeida Costa" in paciente_001["registros"]["cadastro"]
assert "PENDENTE" in paciente_001["registros"]["exames"]

print("[OK] Acesso autorizado.")
print("\nEXAMES:")
print(paciente_001["registros"]["exames"])
print("\nFONTES:")
print(paciente_001["fontes"])


# TESTE 2 — Acesso não autorizado.
try:
    recuperar_paciente(
        ator="clinico_demo_01",
        patient_id="PAC-DEMO-002"
    )
except PermissionError:
    print("\n[OK] Acesso indevido bloqueado.")
else:
    raise AssertionError("Falha no controle de acesso.")


# TESTE 3 — Último paciente.
paciente_1000 = recuperar_paciente(
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-1000"
)

assert len(paciente_1000["fontes"]) == 3

print("[OK] PAC-DEMO-1000 recuperado.")
print("\n[OK] Testes de recuperação concluídos.")

[OK] Acesso autorizado.

EXAMES:
Prontuário: PAC-DEMO-001
Exames:
EX-0001-01: Hemograma | Status: PENDENTE
EX-0001-02: Glicemia | Status: CONCLUÍDO

FONTES:
[{'documento': 'cadastro::PAC-DEMO-001', 'tipo': 'cadastro', 'paciente': 'PAC-DEMO-001'}, {'documento': 'notas::PAC-DEMO-001', 'tipo': 'notas', 'paciente': 'PAC-DEMO-001'}, {'documento': 'exames::PAC-DEMO-001', 'tipo': 'exames', 'paciente': 'PAC-DEMO-001'}]

[OK] Acesso indevido bloqueado.
[OK] PAC-DEMO-1000 recuperado.

[OK] Testes de recuperação concluídos.


### Agente RAG com Qwen

O agente consulta prontuários sintéticos de pacientes autorizados no ChromaDB.

Para exames, retorna os registros diretamente. Para sintomas e observações clínicas, apresenta a transcrição dos trechos recuperados, evitando alterar termos existentes nas notas.

Nas demais consultas narrativas, o Qwen original pode gerar texto com base no contexto recuperado. Se houver um rascunho do Fine-Tuning, ele é apenas auxiliar, não uma fonte clínica. As respostas geradas não possuem validação clínica.


In [22]:
import torch
import unicodedata
from transformers import AutoTokenizer, AutoModelForCausalLM

# Modelo original utilizado exclusivamente pelo RAG.
MODELO_RAG = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_rag = AutoTokenizer.from_pretrained(MODELO_RAG)

modelo_rag = AutoModelForCausalLM.from_pretrained(
    MODELO_RAG,
    dtype=torch.float32
).to("cpu")

modelo_rag.eval()

print("[OK] Qwen original carregado para o RAG.")


def gerar_texto_rag(mensagens, max_new_tokens=110):
    """
    Gera uma resposta com o Qwen a partir das mensagens
    e do contexto recuperado.
    """

    # Aplica o template de conversação do Qwen.
    prompt = tokenizer_rag.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converte o prompt em tensores PyTorch.
    entradas = tokenizer_rag(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cpu")

    with torch.inference_mode():
        saidas = modelo_rag.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_rag.eos_token_id
        )

    # Remove os tokens pertencentes ao prompt.
    tamanho_entrada = entradas["input_ids"].shape[1]

    resposta = tokenizer_rag.decode(
        saidas[0][tamanho_entrada:],
        skip_special_tokens=True
    )

    return resposta.strip()


def responder_rag(pergunta, ator, patient_id, resposta_ft=None):
    """
    Responde perguntas sobre um paciente autorizado.
    Recupera dados no ChromaDB antes de gerar respostas.
    Um rascunho do FT pode apoiar a redação, mas nunca é fonte clínica.
    """

    # 1. Autoriza o acesso antes de qualquer consulta.
    dados = recuperar_paciente(
        ator=ator,
        patient_id=patient_id
    )

    pergunta_normalizada = pergunta.lower()

    # 2. Recuperação exata para exames.
    if any(
        termo in pergunta_normalizada
        for termo in ["exame", "hemograma", "glicemia"]
    ):
        texto_exames = dados["registros"]["exames"]

        if "pendente" in pergunta_normalizada:
            linhas = [
                linha
                for linha in texto_exames.splitlines()
                if "Status: PENDENTE" in linha
            ]

            resposta = (
                "\n".join(linhas)
                if linhas
                else "Nenhum exame pendente registrado."
            )
        else:
            resposta = texto_exames

        return {
            "resposta": resposta,
            "status": "recuperacao_exata",
            "fontes": [f"exames::{patient_id}"]
        }

    # 3. Busca semântica das anotações do paciente.
    documentos = vectorstore_pacientes.similarity_search(
        query=pergunta,
        k=1,
        filter={
            "$and": [
                {"patient_id": patient_id},
                {"record_type": "notas"}
            ]
        }
    )

    if not documentos:
        return {
            "resposta": "Não foram encontradas anotações clínicas.",
            "status": "sem_contexto",
            "fontes": []
        }

    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos
    )

    # Consultas de sintomas/observações usam EXTRATOS LITERAIS das notas.
    # Evita que a LLM invente ou deforme termos clínicos (ex.: cefaleia).
    # O caminho FT -> RAG permanece; o rascunho FT não substitui a fonte.
    pergunta_sem_acentos = unicodedata.normalize(
        "NFKD", pergunta_normalizada
    )
    pergunta_sem_acentos = "".join(
        c for c in pergunta_sem_acentos if not unicodedata.combining(c)
    )
    consulta_de_notas = any(
        termo in pergunta_sem_acentos
        for termo in ("sintoma", "queixa", "observac", "anotac", "relato clinico")
    )
    if consulta_de_notas:
        trechos = []
        for documento in documentos:
            for linha in documento.page_content.splitlines():
                # A base sintética identifica cada observação como NOTA-0001.
                if re.match(r"^\s*NOTA-\d+\s*:", linha):
                    trecho = linha.split(":", 1)[1].strip()
                    if trecho and trecho not in trechos:
                        trechos.append(trecho)
        if trechos:
            return {
                "resposta": "Observações clínicas registradas (transcrição):\n" +
                           "\n".join(f"- {trecho}" for trecho in trechos),
                "status": "recuperacao_textual",
                "fontes": [f"notas::{patient_id}"]
            }
        # Não há registro extraível: não inventamos uma resposta narrativa.
        return {
            "resposta": "Não foi possível extrair uma observação clínica da fonte recuperada.",
            "status": "sem_contexto",
            "fontes": [f"notas::{patient_id}"]
        }

    # Quando o LangGraph percorre FT -> RAG, o texto do FT serve apenas
    # como sugestão de redação. Ele pode conter erros e NÃO é uma fonte.
    apoio_ft = (
        "\n\nRASCUNHO DO MODELO AJUSTADO (NÃO É FONTE CLÍNICA):\n"
        + resposta_ft[:800]
        + "\nIgnore qualquer afirmação sem apoio explícito nas notas acima."
        if resposta_ft else ""
    )

    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um assistente de consulta a prontuários "
                "sintéticos. Responda em português, de forma breve, "
                "utilizando exclusivamente as informações do contexto. "
                "Não invente sintomas, exames ou resultados. "
                "Não faça diagnósticos nem prescrições. "
                "Se não houver informação suficiente, diga isso. "
                "Se houver um rascunho de outro modelo, trate-o como "
                "não confiável e descarte fatos não presentes nas notas."
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta}{apoio_ft}"
            )
        }
    ]

    resposta = gerar_texto_rag(mensagens)

    return {
        "resposta": resposta,
        "status": "geracao_nao_validada",
        "fontes": [
            f"notas::{patient_id}"
        ]
    }


# =========================================================
# TESTE 1 — Recuperação de exames
# =========================================================

print("\nTESTE 1 — EXAMES PENDENTES")

r1 = responder_rag(
    pergunta="Quais exames estão pendentes?",
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

print("Resposta:", r1["resposta"])
print("Fontes:", r1["fontes"])
print("Status:", r1["status"])

assert "Hemograma" in r1["resposta"]
assert "Glicemia" not in r1["resposta"]


# =========================================================
# TESTE 2 — Interpretação das observações clínicas
# =========================================================

print("\nTESTE 2 — SINTOMAS REGISTRADOS")

r2 = responder_rag(
    pergunta="Quais sintomas estão registrados?",
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

print("Resposta:", r2["resposta"])
print("Fontes:", r2["fontes"])
print("Status:", r2["status"])
assert r2["status"] == "recuperacao_textual"
assert "cefaleia" in r2["resposta"].lower()
assert "tontura" in r2["resposta"].lower()


# =========================================================
# TESTE 3 — Bloqueio de acesso indevido
# =========================================================

print("\nTESTE 3 — ACESSO NÃO AUTORIZADO")

try:
    responder_rag(
        pergunta="Quais sintomas estão registrados?",
        ator="clinico_demo_01",
        patient_id="PAC-DEMO-002"
    )
except PermissionError:
    print("[OK] Acesso indevido bloqueado.")
else:
    raise AssertionError("Falha no controle de acesso.")


print("\n[OK] Testes do agente RAG finalizados.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[OK] Qwen original carregado para o RAG.

TESTE 1 — EXAMES PENDENTES
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']
Status: recuperacao_exata

TESTE 2 — SINTOMAS REGISTRADOS
Resposta: Observações clínicas registradas (transcrição):
- Paciente relata episódios de cefaleia e tontura.
Fontes: ['notas::PAC-DEMO-001']
Status: recuperacao_textual

TESTE 3 — ACESSO NÃO AUTORIZADO
[OK] Acesso indevido bloqueado.

[OK] Testes do agente RAG finalizados.


## 5. LangGraph e integração dos agentes

### Orquestração dos agentes com LangGraph

O LangGraph direciona as consultas por três caminhos demonstrativos:

- **RAG direto:** consultas exatas a registros de pacientes autorizados.
- **Fine-Tuning → RAG:** consultas narrativas sobre pacientes autorizados; o RAG consulta as fontes e determina a resposta final.
- **Bloqueio:** prescrições proibidas, identificação insuficiente ou acesso não autorizado.

Perguntas médicas gerais também passam pelo nó Fine-Tuning e atravessam o nó RAG sem recuperar prontuários. Depois, as solicitações passam pela validação de saída e pela auditoria.

Nesta etapa, o agente Fine-Tuning ainda é provisório; ele será substituído pela implementação com os pesos ajustados após o carregamento do modelo. As regras de segurança são demonstrativas e não substituem controles hospitalares reais.


In [23]:
import json
import re
import unicodedata
from uuid import uuid4
from datetime import datetime, timezone
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class EstadoAssistente(TypedDict, total=False):
    pergunta: str
    ator: str
    request_id: str
    patient_id: str
    rota: str
    resposta: str
    resposta_ft: str  # Rascunho experimental do FT, nunca fonte clínica.
    status: str
    fontes: list
    trilha: list


def normalizar_texto(texto):
    """Remove acentos e converte o texto para minúsculas."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def verificar_entrada(state):
    """Identifica paciente e autorização ANTES de qualquer acesso a dados."""
    pergunta = normalizar_texto(state["pergunta"])
    trilha = state.get("trilha", []) + ["verificar_entrada"]

    # Bloqueios demonstrativos permanecem ANTES dos agentes.
    if re.search(
        r"\b(prescreva|receite|diagnostique|posologia|dosagem|qual a dose)\b",
        pergunta
    ):
        return {
            "rota": "bloqueio", "status": "solicitacao_bloqueada",
            "resposta": "Não realizo prescrições, definição de doses ou diagnósticos autônomos.",
            "trilha": trilha
        }

    match_id = re.search(r"\bPAC-DEMO-\d{3,4}\b", state["pergunta"], re.I)
    patient_id = match_id.group(0).upper() if match_id else None

    if patient_id is None:
        encontrados = []
        for paciente in pacientes:
            nome = normalizar_texto(paciente["nome"])
            if re.search(r"(?<!\w)" + re.escape(nome) + r"(?!\w)", pergunta):
                encontrados.append(paciente)

        if encontrados:
            match_idade = re.search(r"\b(\d{1,3})\s*anos?\b", pergunta)
            if match_idade:
                idade = int(match_idade.group(1))
                encontrados = [p for p in encontrados if p["idade"] == idade]

            if len(encontrados) > 1:
                return {
                    "rota": "bloqueio", "status": "nome_ambiguo",
                    "resposta": "Existem múltiplos registros com esse nome. Informe também a idade ou o ID do prontuário.",
                    "trilha": trilha
                }
            if len(encontrados) == 1:
                patient_id = encontrados[0]["patient_id"]
            else:
                return {
                    "rota": "bloqueio", "status": "paciente_nao_identificado",
                    "resposta": "Não foi possível identificar o paciente com os dados informados.",
                    "trilha": trilha
                }

    # Nunca envia uma consulta identificada e não autorizada para o modelo.
    if patient_id is not None:
        if (state["ator"], patient_id) not in permissoes_demo:
            return {
                "rota": "bloqueio", "status": "acesso_negado",
                "resposta": "Acesso ao prontuário não autorizado.",
                "trilha": trilha
            }

        # Campos estruturados têm recuperação exata, sem precisar gerar texto.
        consulta_exata = any(termo in pergunta for termo in (
            "exame", "hemograma", "glicemia"
        ))
        return {
            "rota": "rag" if consulta_exata else "fine_tuning_rag",
            "patient_id": patient_id,
            "trilha": trilha
        }

    # Proíbe consultas a prontuário sem identificação confirmada.
    if any(termo in pergunta for termo in (
        "paciente", "prontuario", "exames pendentes", "exames do", "sintomas registrados"
    )):
        return {
            "rota": "bloqueio", "status": "paciente_nao_identificado",
            "resposta": "Não foi possível identificar o paciente. Informe dados suficientes para diferenciá-lo.",
            "trilha": trilha
        }

    # Perguntas médicas gerais entram no FT. O nó RAG seguinte não procura
    # prontuários se não houver paciente identificado/autorizado.
    return {"rota": "fine_tuning", "trilha": trilha}


def escolher_rota(state):
    return state["rota"]


def agente_rag(state):
    """RAG direto ou RAG após FT; sem patient_id não consulta prontuários."""
    trilha = state["trilha"] + ["agente_rag"]
    patient_id = state.get("patient_id")

    if not patient_id:
        # Pergunta geral: nó de passagem, SEM recuperação artificial de dados.
        return {
            "resposta": state.get("resposta_ft") or state.get("resposta", ""),
            "status": state.get("status", "sem_resposta"),
            "fontes": [], "trilha": trilha
        }

    resultado = responder_rag(
        pergunta=state["pergunta"],
        ator=state["ator"],
        patient_id=patient_id,
        resposta_ft=state.get("resposta_ft")
    )
    return {
        "resposta": resultado["resposta"], "status": resultado["status"],
        "fontes": resultado["fontes"], "trilha": trilha
    }


def agente_fine_tuning(state):
    """Implementação provisória, substituída na integração do modelo ajustado."""
    return {
        "resposta_ft": "",
        "resposta": "O modelo com Fine-Tuning ainda não foi restaurado neste ambiente.",
        "status": "fine_tuning_pendente", "fontes": [],
        "trilha": state["trilha"] + ["agente_fine_tuning"]
    }


def agente_bloqueio(state):
    return {"fontes": [], "trilha": state["trilha"] + ["agente_bloqueio"]}


def validar_saida(state):
    """Segurança pós-agentes; não substitui validação clínica."""
    resposta = (state.get("resposta") or "").strip()
    status = state.get("status", "sem_resposta")
    fontes = state.get("fontes", [])

    if not resposta:
        resposta = "Não foi possível produzir uma resposta; revisão humana necessária."
        status = "sem_resposta"

    # Bloqueio demonstrativo de instruções explícitas de dosagem na saída.
    if status in ("geracao_nao_validada", "recuperacao_exata", "recuperacao_textual") and re.search(
        r"\b(?:tome|administre|aplique|ingerir)\s+\d+(?:[.,]\d+)?\s*(?:mg|ml|comprimidos?)\b",
        normalizar_texto(resposta), re.I
    ):
        resposta = "Resposta bloqueada por possível orientação medicamentosa. Solicite revisão profissional."
        status = "resposta_bloqueada"
        fontes = []
    elif status == "geracao_nao_validada":
        resposta += "\n\n[Aviso: conteúdo gerado por IA, sem validação clínica. Revisão profissional necessária.]"

    if state.get("rota") == "bloqueio":
        fontes = []

    return {
        "resposta": resposta, "status": status, "fontes": fontes,
        "trilha": state["trilha"] + ["validar_saida"]
    }


def registrar_auditoria(state):
    """Registra somente a rota e o percurso, sem perguntas nem prontuários."""
    trilha = state["trilha"] + ["registrar_auditoria"]
    registro = {
        "request_id": state["request_id"],
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "rota": state["rota"], "status": state["status"],
        "caminho": trilha,
        "quantidade_fontes": len(state.get("fontes", []))
    }
    arquivo = RESULTS_PATH / "auditoria_langgraph.jsonl"
    with open(arquivo, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    return {"trilha": trilha}


def montar_grafo():
    """Input -> FT -> RAG -> Security ou Input -> RAG -> Security."""
    workflow = StateGraph(EstadoAssistente)
    workflow.add_node("verificar_entrada", verificar_entrada)
    workflow.add_node("agente_rag", agente_rag)
    workflow.add_node("agente_fine_tuning", agente_fine_tuning)
    workflow.add_node("agente_bloqueio", agente_bloqueio)
    workflow.add_node("validar_saida", validar_saida)
    workflow.add_node("registrar_auditoria", registrar_auditoria)

    workflow.add_edge(START, "verificar_entrada")
    workflow.add_conditional_edges(
        "verificar_entrada", escolher_rota,
        {
            "rag": "agente_rag",
            "fine_tuning_rag": "agente_fine_tuning",
            "fine_tuning": "agente_fine_tuning",
            "bloqueio": "agente_bloqueio"
        }
    )
    workflow.add_edge("agente_fine_tuning", "agente_rag")
    workflow.add_edge("agente_rag", "validar_saida")
    workflow.add_edge("agente_bloqueio", "validar_saida")
    workflow.add_edge("validar_saida", "registrar_auditoria")
    workflow.add_edge("registrar_auditoria", END)
    return workflow.compile()


grafo = montar_grafo()


def consultar_assistente(pergunta, ator="clinico_demo_01"):
    """Executa pergunta pelo LangGraph completo."""
    return grafo.invoke({
        "pergunta": pergunta, "ator": ator,
        "request_id": str(uuid4()), "trilha": []
    })


print("[OK] Grafo preparado: RAG direto ou Fine-Tuning -> RAG, segurança e auditoria.")
print("[INFO] O nó Fine-Tuning real será conectado após carregar o modelo ajustado.")


[OK] Grafo preparado: RAG direto ou Fine-Tuning -> RAG, segurança e auditoria.
[INFO] O nó Fine-Tuning real será conectado após carregar o modelo ajustado.


### Restauração do modelo ajustado com Fine-Tuning

Restauramos os pesos do Qwen2.5-0.5B-Instruct treinado com MedQuAD, a partir do ZIP exportado no Full Fine-Tuning definitivo.

Esta etapa não treina novamente o modelo nem altera seus pesos. O carregamento para inferência acontece na próxima etapa.


In [24]:

from pathlib import Path
import zipfile

# Pasta onde o modelo será restaurado no Colab.
PASTA_MODELOS = Path("/content/models")
PASTA_FT = PASTA_MODELOS / "qwen-medquad-full-ft-v2"

# Procura o ZIP nos locais conhecidos.
possiveis_zips = [
    BASE_PATH / "models" / "qwen-medquad-full-ft-v2.zip",
    BASE_PATH / "qwen-medquad-full-ft-v2.zip",
    Path("/content/qwen-medquad-full-ft-v2.zip")
]

zip_ft = next(
    (arquivo for arquivo in possiveis_zips if arquivo.is_file()),
    None
)

# Evita extrair novamente se o modelo já estiver completo.
modelo_restaurado = (
    (PASTA_FT / "config.json").is_file()
    and bool(list(PASTA_FT.glob("*.safetensors")))
)

if modelo_restaurado:
    print("[OK] Modelo já estava extraído.")

else:
    if zip_ft is None:
        raise FileNotFoundError(
            "ZIP não encontrado. Envie "
            "qwen-medquad-full-ft-v2.zip "
            "para a pasta TechChallenge3/models do Drive."
        )

    print("[OK] ZIP encontrado:", zip_ft)
    print("Restaurando os pesos do Fine-Tuning...")

    PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_ft, "r") as arquivo:
        nomes = arquivo.namelist()

        # Confirma a estrutura do ZIP que geramos
        # ao exportar o modelo treinado.
        assert (
            "qwen-medquad-full-ft-v2/config.json" in nomes
        ), "Estrutura do ZIP diferente da esperada."

        # Impede a extração de caminhos inseguros.
        for membro in arquivo.infolist():
            caminho = Path(membro.filename)

            if caminho.is_absolute() or ".." in caminho.parts:
                raise ValueError(
                    "Caminho inseguro encontrado no ZIP."
                )

        arquivo.extractall(PASTA_MODELOS)

# Verifica os arquivos essenciais.
assert (PASTA_FT / "config.json").is_file(), (
    "Configuração do modelo não encontrada."
)

assert list(PASTA_FT.glob("*.safetensors")), (
    "Pesos do Fine-Tuning não encontrados."
)

print("\n[OK] Modelo Fine-Tuning restaurado!")
print("Diretório:", PASTA_FT)
print("Arquivos disponíveis:")

for arquivo in sorted(PASTA_FT.iterdir()):
    if arquivo.is_file():
        print("-", arquivo.name)


[OK] ZIP encontrado: /content/drive/MyDrive/TechChallenge3/models/qwen-medquad-full-ft-v2.zip
Restaurando os pesos do Fine-Tuning...

[OK] Modelo Fine-Tuning restaurado!
Diretório: /content/models/qwen-medquad-full-ft-v2
Arquivos disponíveis:
- chat_template.jinja
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin
- training_metrics.json


### Carregamento e teste do modelo com Fine-Tuning

Nesta etapa, carregamos o Qwen2.5-0.5B-Instruct ajustado
com o dataset MedQuAD.

Os pesos foram recuperados do treinamento Full Fine-Tuning
realizado anteriormente, sem necessidade de novo treinamento.

Realizamos uma inferência com uma pergunta médica geral
para verificar o funcionamento do modelo ajustado.

A resposta será analisada posteriormente, pois uma geração
bem-sucedida não comprova precisão ou segurança clínica.

In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Diretório restaurado na etapa anterior.
MODELO_FT_PATH = PASTA_FT

# 1. Carrega o tokenizer do nosso modelo ajustado.
tokenizer_ft = AutoTokenizer.from_pretrained(
    str(MODELO_FT_PATH)
)

# 2. Carrega os pesos resultantes do Full Fine-Tuning.
# Utilizamos CPU para não depender da disponibilidade de GPU.
modelo_ft = AutoModelForCausalLM.from_pretrained(
    str(MODELO_FT_PATH),
    dtype=torch.float32,
    low_cpu_mem_usage=True
).to("cpu")

# Modo de inferência: não atualiza os pesos.
modelo_ft.eval()

print("[OK] Modelo Fine-Tuning carregado.")
print("Dispositivo:", next(modelo_ft.parameters()).device)


def gerar_resposta_ft(pergunta, max_new_tokens=120):
    """
    Gera uma resposta usando exclusivamente
    o Qwen ajustado com MedQuAD.
    """

    mensagens = [
        {
            "role": "system",
            "content": (
                "You are a medical information assistant. "
                "Provide a concise, factual answer. "
                "Do not invent information or prescribe treatments. "
                "If uncertain, acknowledge uncertainty."
            )
        },
        {
            "role": "user",
            "content": pergunta
        }
    ]

    # Aplica o template original de conversação do Qwen.
    prompt = tokenizer_ft.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokeniza a entrada corretamente.
    entradas = tokenizer_ft(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cpu")

    # Geração determinística, sem gradientes.
    with torch.inference_mode():
        saidas = modelo_ft.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_ft.eos_token_id
        )

    # Remove os tokens correspondentes à pergunta.
    tamanho_prompt = entradas["input_ids"].shape[1]

    resposta = tokenizer_ft.decode(
        saidas[0][tamanho_prompt:],
        skip_special_tokens=True
    )

    return resposta.strip()


# =====================================================
# TESTE — Pergunta médica geral
# =====================================================

pergunta_teste = "What is diabetes mellitus?"

print("\nPERGUNTA:", pergunta_teste)

resposta_teste = gerar_resposta_ft(pergunta_teste)

print("\nRESPOSTA DO MODELO AJUSTADO:")
print(resposta_teste)

print("\n[OK] Teste de inferência finalizado.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[OK] Modelo Fine-Tuning carregado.
Dispositivo: cpu

PERGUNTA: What is diabetes mellitus?

RESPOSTA DO MODELO AJUSTADO:
Diabetes mellitus (DM) is a chronic disease that occurs when the body does not make enough insulin or cannot use insulin effectively. Insulin is a hormone made by the pancreas, an organ located behind the stomach. The pancreas releases insulin into the bloodstream to help control blood sugar levels. When people with DM do not have enough insulin, their blood glucose levels rise above normal and can cause serious health problems such as heart attack, stroke, vision loss, kidney failure, amputation of feet and legs, and death. People with DM may also develop other complications including high blood pressure, nerve

[OK] Teste de inferência finalizado.


### Integração do modelo Fine-Tuning ao LangGraph

Substituímos o nó provisório pelo Qwen ajustado com MedQuAD e recompilamos o LangGraph, sem repetir o treinamento.

Para perguntas médicas gerais, o modelo ajustado produz a resposta, sem consulta a prontuários. Nas consultas narrativas sobre um paciente autorizado, seu texto é um rascunho auxiliar; o agente RAG consulta as notas e fundamenta a resposta final nos registros recuperados.

As gerações que não forem transcrições diretas de registros são identificadas como não validadas clinicamente e exigem revisão profissional.


In [26]:
# =========================================================
# INTEGRAÇÃO DOS PESOS AJUSTADOS AO LANGGRAPH
# =========================================================

def agente_fine_tuning(state):
    """FT prepara resposta geral/rascunho; RAG verifica dados do paciente."""
    pergunta = state["pergunta"]
    if state.get("patient_id"):
        # Não envia ao FT os registros recuperados nem dados do ChromaDB.
        # O FT fornece apenas um rascunho geral, NÃO fatos do prontuário.
        nome = next(
            (p["nome"] for p in pacientes if p["patient_id"] == state["patient_id"]),
            ""
        )
        if nome:
            pergunta = re.sub(re.escape(nome), "um paciente fictício", pergunta, flags=re.I)

    rascunho = gerar_resposta_ft(
        pergunta=pergunta,
        max_new_tokens=80 if state.get("patient_id") else 160
    )
    status = "geracao_nao_validada" if rascunho else "sem_resposta"
    resposta = rascunho or "O modelo não conseguiu gerar uma resposta."
    # Em consultas com paciente, o nó RAG vai substituir esta resposta
    # pelo texto limitado ao contexto realmente recuperado.
    return {
        "resposta_ft": rascunho,
        "resposta": resposta,
        "status": status,
        "fontes": [],
        "trilha": state["trilha"] + ["agente_fine_tuning"]
    }


# A função montar_grafo() definida na orquestração preserva as arestas;
# recompilamos somente para conectar o nó Fine-Tuning real.
grafo = montar_grafo()
print("[OK] Grafo recompilado com modelo FT real -> RAG -> segurança.")


# TESTE: pergunta médica geral. O nó RAG é atravessado SEM consulta
# a prontuários, porque não existe paciente identificado.
resultado_ft = consultar_assistente("What is diabetes mellitus?")
print("Rota:", resultado_ft["rota"])
print("Status:", resultado_ft["status"])
print("Caminho:", " -> ".join(resultado_ft["trilha"]))
print("Resposta:", resultado_ft["resposta"])
print("Fontes:", resultado_ft["fontes"])

assert resultado_ft["rota"] == "fine_tuning"
assert resultado_ft["status"] in ("geracao_nao_validada", "resposta_bloqueada")
assert resultado_ft["trilha"].index("agente_fine_tuning") < resultado_ft["trilha"].index("agente_rag")
assert resultado_ft["fontes"] == []
assert resultado_ft["trilha"][-2:] == ["validar_saida", "registrar_auditoria"]
print("[OK] Qwen ajustado integrado; nenhum prontuário foi consultado.")


[OK] Grafo recompilado com modelo FT real -> RAG -> segurança.
Rota: fine_tuning
Status: geracao_nao_validada
Caminho: verificar_entrada -> agente_fine_tuning -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: Diabetes mellitus (DM) is a chronic disease that occurs when the body does not make enough insulin or cannot use insulin effectively. Insulin is a hormone made by the pancreas, an organ located behind the stomach. The pancreas releases insulin into the bloodstream to help control blood sugar levels. When people with DM do not have enough insulin, their blood glucose levels rise above normal and can cause serious health problems such as heart attack, stroke, vision loss, kidney failure, amputation of feet and legs, and death. People with DM may also develop other complications including high blood pressure, nerve damage in the hands and feet, and skin problems. Diabetes mellitus is one type of metabolic syndrome. Metabolic syndrome is a group of conditions that occur 

## 6. Validação e demonstração

### Validação final do assistente médico

Nesta etapa, testamos os caminhos do LangGraph e a auditoria com oito solicitações:

1. Exames por identificador do paciente (RAG direto).
2. Exames por nome completo e idade (RAG direto).
3. Sintomas do paciente (Fine-Tuning → RAG, com transcrição da nota recuperada).
4. Nome duplicado sem informação suficiente (bloqueio).
5. Paciente não identificado (bloqueio).
6. Acesso não autorizado (bloqueio).
7. Pedido de prescrição (bloqueio).
8. Pergunta médica geral com o modelo ajustado (sem recuperar prontuário).

Verificamos também as fontes consultadas e os registros de auditoria. Os testes demonstram funcionamento do protótipo, não validação clínica ou segurança para produção.


In [27]:

# =========================================================
# VALIDAÇÃO FINAL DO ASSISTENTE MÉDICO
# =========================================================

import json
from collections import Counter


def executar_teste(nome, pergunta, rota, status):
    """
    Executa um teste pelo LangGraph e verifica
    o roteamento, o status e a auditoria.
    """
    resultado = consultar_assistente(pergunta)

    assert resultado["rota"] == rota, (
        f"Rota incorreta no teste: {nome}"
    )
    assert resultado["status"] == status, (
        f"Status incorreto no teste: {nome}"
    )
    assert resultado["trilha"][-2:] == ["validar_saida", "registrar_auditoria"]

    print(f"\n{'=' * 55}")
    print("TESTE:", nome)
    print("Pergunta:", pergunta)
    print("Rota:", resultado["rota"])
    print("Status:", resultado["status"])
    print("Caminho:", " -> ".join(resultado["trilha"]))
    print("Resposta:", resultado["resposta"])
    print("Fontes:", resultado.get("fontes", []))

    return resultado


# =========================================================
# TESTE 1 — CONSULTA PELO IDENTIFICADOR
# =========================================================

teste_id = executar_teste(
    nome="Consulta pelo ID",
    pergunta="Quais exames estão pendentes para PAC-DEMO-001?",
    rota="rag",
    status="recuperacao_exata"
)

assert "Hemograma" in teste_id["resposta"]
assert "Glicemia" not in teste_id["resposta"]


# =========================================================
# TESTE 2 — CONSULTA PELO NOME COMPLETO
# =========================================================

paciente_demo = next(
    p for p in pacientes
    if p["patient_id"] == "PAC-DEMO-001"
)

pergunta_nome = (
    f"Quais exames estão pendentes para a paciente "
    f"{paciente_demo['nome']}, "
    f"{paciente_demo['idade']} anos?"
)

teste_nome = executar_teste(
    nome="Consulta pelo nome",
    pergunta=pergunta_nome,
    rota="rag",
    status="recuperacao_exata"
)

assert "Hemograma" in teste_nome["resposta"]
assert "agente_fine_tuning" not in teste_nome["trilha"]

print("[OK] Nome convertido para prontuário autorizado.")


# =========================================================
# TESTE 2B — RESPOSTA NARRATIVA: FINE-TUNING -> RAG
# =========================================================

# O mesmo paciente autorizado é identificado por nome e idade.
# O FT gera um rascunho geral; o RAG consulta as notas e fundamenta
# a resposta final nos dados efetivamente recuperados.
teste_combinado = executar_teste(
    nome="Sintomas: Fine-Tuning seguido de RAG",
    pergunta=(
        f"Quais sintomas estão registrados para a paciente "
        f"{paciente_demo['nome']}, {paciente_demo['idade']} anos?"
    ),
    rota="fine_tuning_rag",
    status="recuperacao_textual"
)

# A resposta textual deve reproduzir os termos da nota original,
# não a paráfrase incorreta observada no teste anterior.
assert "cefaleia" in teste_combinado["resposta"].lower()
assert "tontura" in teste_combinado["resposta"].lower()
assert "cefalorraquina" not in teste_combinado["resposta"].lower()
assert "Paciente relata episódios de cefaleia e tontura." in teste_combinado["resposta"]

assert teste_combinado["trilha"].index("agente_fine_tuning") < (
    teste_combinado["trilha"].index("agente_rag")
)
assert teste_combinado["fontes"] == ["notas::PAC-DEMO-001"]
assert "agente_bloqueio" not in teste_combinado["trilha"]


# =========================================================
# TESTE 3 — NOMES DUPLICADOS
# =========================================================

contagem_nomes = Counter(
    p["nome"] for p in pacientes
)

nomes_duplicados = [
    nome for nome, quantidade in contagem_nomes.items()
    if quantidade > 1
]

if nomes_duplicados:
    nome_duplicado = nomes_duplicados[0]

    teste_duplicado = executar_teste(
        nome="Nome duplicado",
        pergunta=(
            f"Quais exames estão pendentes para "
            f"{nome_duplicado}?"
        ),
        rota="bloqueio",
        status="nome_ambiguo"
    )

    assert "agente_rag" not in teste_duplicado["trilha"]
else:
    print("\n[INFO] Nenhum nome duplicado na base.")


# =========================================================
# TESTE 4 — PACIENTE NÃO IDENTIFICADO
# =========================================================

teste_ausente = executar_teste(
    nome="Paciente não identificado",
    pergunta="Quais exames estão pendentes para o paciente Eric?",
    rota="bloqueio",
    status="paciente_nao_identificado"
)

assert "agente_rag" not in teste_ausente["trilha"]


# =========================================================
# TESTE 5 — ACESSO NÃO AUTORIZADO
# =========================================================

teste_acesso = executar_teste(
    nome="Acesso não autorizado",
    pergunta="Quais exames existem para PAC-DEMO-002?",
    rota="bloqueio",
    status="acesso_negado"
)

assert "agente_rag" not in teste_acesso["trilha"]
assert teste_acesso.get("fontes", []) == []


# =========================================================
# TESTE 6 — SOLICITAÇÃO DE PRESCRIÇÃO
# =========================================================

teste_prescricao = executar_teste(
    nome="Solicitação de prescrição",
    pergunta="Prescreva um medicamento para PAC-DEMO-001",
    rota="bloqueio",
    status="solicitacao_bloqueada"
)

assert "agente_fine_tuning" not in teste_prescricao["trilha"]
assert "agente_rag" not in teste_prescricao["trilha"]


# =========================================================
# TESTE 7 — MODELO COM FINE-TUNING
# =========================================================

# Reutiliza o teste real realizado na integração do modelo ajustado,
# evitando gerar novamente uma resposta em CPU.

assert resultado_ft["rota"] == "fine_tuning"
assert resultado_ft["status"] in ("geracao_nao_validada", "resposta_bloqueada")
assert "agente_fine_tuning" in resultado_ft["trilha"]
assert "agente_rag" in resultado_ft["trilha"]  # Sem recuperação: fontes vazias

print(f"\n{'=' * 55}")
print("TESTE: Fine-Tuning integrado")
print("Rota:", resultado_ft["rota"])
print("Status:", resultado_ft["status"])
print("Caminho:", " -> ".join(resultado_ft["trilha"]))
print("[OK] Modelo ajustado executado na etapa de integração.")


# =========================================================
# TESTE 8 — AUDITORIA
# =========================================================

arquivo_auditoria = (
    RESULTS_PATH / "auditoria_langgraph.jsonl"
)

assert arquivo_auditoria.exists()

with open(arquivo_auditoria, "r", encoding="utf-8") as f:
    registros = [
        json.loads(linha)
        for linha in f
        if linha.strip()
    ]

# Confirma os registros desta execução,
# incluindo o teste de Fine-Tuning da etapa de integração.

resultados = [
    teste_id,
    teste_nome,
    teste_combinado,
    teste_ausente,
    teste_acesso,
    teste_prescricao,
    resultado_ft
]

if nomes_duplicados:
    resultados.append(teste_duplicado)

ids_testados = {
    resultado["request_id"]
    for resultado in resultados
}

registros_testados = {
    registro["request_id"]: registro
    for registro in registros
    if registro["request_id"] in ids_testados
}

assert set(registros_testados) == ids_testados

for registro in registros_testados.values():
    assert registro["caminho"][-2:] == ["validar_saida", "registrar_auditoria"]

    # O log não deve registrar o texto das perguntas
    # nem das respostas clínicas.
    assert "pergunta" not in registro
    assert "resposta" not in registro

print(f"\n{'=' * 55}")
print("[OK] Auditoria confirmada.")
print("Registros desta validação:", len(registros_testados))
print("Arquivo:", arquivo_auditoria)


# =========================================================
# RESULTADO FINAL
# =========================================================

print("\n" + "=" * 55)
print("[OK] VALIDAÇÃO FUNCIONAL CONCLUÍDA!")
print("=" * 55)

print("[OK] Consultas por ID e nome completo (RAG direto)")
print("[OK] Consulta narrativa: Fine-Tuning -> RAG -> segurança")
print("[OK] Tratamento de nomes duplicados")
print("[OK] Identificação de pacientes ausentes")
print("[OK] Controle de acesso demonstrativo")
print("[OK] Bloqueio de prescrição direta")
print("[OK] Fine-Tuning integrado")
print("[OK] Auditoria dos caminhos executados")



TESTE: Consulta pelo ID
Pergunta: Quais exames estão pendentes para PAC-DEMO-001?
Rota: rag
Status: recuperacao_exata
Caminho: verificar_entrada -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']

TESTE: Consulta pelo nome
Pergunta: Quais exames estão pendentes para a paciente Marina Almeida Costa, 53 anos?
Rota: rag
Status: recuperacao_exata
Caminho: verificar_entrada -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']
[OK] Nome convertido para prontuário autorizado.

TESTE: Sintomas: Fine-Tuning seguido de RAG
Pergunta: Quais sintomas estão registrados para a paciente Marina Almeida Costa, 53 anos?
Rota: fine_tuning_rag
Status: recuperacao_textual
Caminho: verificar_entrada -> agente_fine_tuning -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: Observações clínicas registradas (transcrição):
- Pacie

### Teste realizando o input no terminal

Digite uma pergunta para executar o grafo já carregado. Esta etapa é interativa e pode ser repetida para demonstrar os caminhos RAG direto e Fine-Tuning → RAG.


In [37]:
from uuid import uuid4

pergunta = input("Digite sua pergunta: ").strip()

resultado = grafo.invoke({
    "pergunta": pergunta,
    "ator": "clinico_demo_01",
    "request_id": str(uuid4()),
    "trilha": []
})

print("\n" + "=" * 55)
print("ASSISTENTE MÉDICO — RESPOSTA")
print("=" * 55)

print(f"\n{resultado.get('resposta', 'Sem resposta disponível.')}")

print("\n" + "-" * 55)
print("DETALHES DA EXECUÇÃO")
print("-" * 55)

print(f"Rota: {resultado.get('rota', 'Não informada')}")
print(f"Status: {resultado.get('status', 'Não informado')}")

trilha = resultado.get("trilha", [])
print(f"Fluxo: {' → '.join(trilha)}")

fontes = resultado.get("fontes", [])
if fontes:
    print("\nFontes consultadas:")
    for fonte in fontes:
        print(f"  - {fonte}")

print(f"\nID da requisição: {resultado.get('request_id', '-')}")
print("=" * 55)

Digite sua pergunta: Quais sintomas estão registrados para Rafael Santos Silva?

ASSISTENTE MÉDICO — RESPOSTA

Acesso ao prontuário não autorizado.

-------------------------------------------------------
DETALHES DA EXECUÇÃO
-------------------------------------------------------
Rota: bloqueio
Status: acesso_negado
Fluxo: verificar_entrada → agente_bloqueio → validar_saida → registrar_auditoria

ID da requisição: 317d8ed4-5009-4985-846c-edb710e8b312
